In [36]:
# 0. Load to HDFS 
# /user/userjuly2025019/spark/dailyHuntDataSet
# 1. Import the values 
import pyspark 
from pyspark.sql import SparkSession 
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

In [37]:
# 2. I need to create a spark session 
spark = SparkSession.builder.master("local")\
        .appName("e2e-demo-dailyhunt")\
        .getOrCreate()

In [3]:
spark

In [39]:
# 3. load the data 
user_profile_df = spark.read.format("csv")\
                    .option("header","true")\
                    .option("inferschema","true")\
                    .load("/user/userjuly2025019/spark/dailyHuntDataSet/user_profile_new.csv")

article_metadata_df = spark.read.format("csv")\
                        .option("header","true")\
                        .option("inferschema","true")\
                        .load("/user/userjuly2025019/spark/dailyHuntDataSet/article_metadata.csv")

user_activity_df = spark.read.format("csv")\
                    .option("header","true")\
                    .option("inferschema","true")\
                    .load("/user/userjuly2025019/spark/dailyHuntDataSet/new_user_activity.csv")


user_profile_df.printSchema()
article_metadata_df.printSchema()
user_activity_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- article_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- publish_date: string (nullable = true)
 |-- author: string (nullable = true)

root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- time_spent: integer (nullable = true)



In [40]:
user_profile_df.show(truncate=False)
# root
#  |-- user_id: string (nullable = true) [correct DT]
#  |-- age: integer (nullable = true) [correct DT]
#  |-- region: string (nullable = true) [correct DT]
#  |-- gender: string (nullable = true) [correct DT]

# TODO:
# make sure to correct the gender mistyped values 

+-------+---+-------+----------+
|user_id|age|region |gender    |
+-------+---+-------+----------+
|user_1 |64 |North  |male      |
|user_2 |64 |East   |Female    |
|user_3 |37 |West   |MALE      |
|user_4 |32 |South  |Female    |
|user_5 |37 |Central|Non-Binary|
|user_6 |22 |South  |MALE      |
|user_7 |35 |East   |Other     |
|user_8 |63 |North  |Non-Binary|
|user_9 |26 |Central|Male      |
|user_10|29 |North  |Non-Binary|
|user_11|30 |North  |Non-Binary|
|user_12|22 |Central|male      |
|user_13|54 |Central|Male      |
|user_14|23 |Central|Non-Binary|
|user_15|40 |North  |Female    |
|user_16|37 |West   |Other     |
|user_17|18 |Central|Other     |
|user_18|20 |Central|Male      |
|user_19|36 |East   |Female    |
|user_20|41 |Central|Non-Binary|
+-------+---+-------+----------+
only showing top 20 rows



In [41]:
article_metadata_df.show()

# root
#  |-- article_id: string (nullable = true) [correct DT]
#  |-- title: string (nullable = true) [correct DT]
#  |-- category: string (nullable = true) [correct DT]
#  |-- publish_date: string (nullable = true) [incorrect DT]
#  |-- author: string (nullable = true) [correct DT]


# TODO:
# publish_date -> change DT to Date

+----------+--------------------+-------------+------------+----------------+
|article_id|               title|     category|publish_date|          author|
+----------+--------------------+-------------+------------+----------------+
| article_1|Sound move bit po...|       Health|  2024-03-05|Michael Robinson|
| article_2|Arm head drop rea...|       Travel|  2024-08-03|   Emily Mcguire|
| article_3|Late exist bill d...|    Education|  2024-02-28|   Anna Williams|
| article_4|Single start skil...|   Technology|  2024-07-28| Diana Hernandez|
| article_5|Suffer security s...|Entertainment|  2024-09-17|   Laurie French|
| article_6|Baby happy fish s...|       Health|  2024-10-20|  Terry Caldwell|
| article_7|Deal our personal...|Entertainment|  2024-08-27|   Matthew Perez|
| article_8|Keep view yes col...|       Travel|  2023-12-17|      Rick Hicks|
| article_9|Analysis together...|       Health|  2024-01-20|     Ray Gilbert|
|article_10|Without experienc...|   Technology|  2024-08-30|    

In [42]:
user_activity_df.show(truncate=False)

# root
#  |-- user_id: string (nullable = true) [correct]
#  |-- article_id: string (nullable = true) [correct]
#  |-- action: string (nullable = true) [correct]
#  |-- timestamp: string (nullable = true) [incorrect]
#  |-- time_spent: integer (nullable = true) [correct]


# TODO: 
# convert timestamp from string to timestamp datatype 

+--------+-----------+------+-------------------+----------+
|user_id |article_id |action|timestamp          |time_spent|
+--------+-----------+------+-------------------+----------+
|user_372|article_73 |read  |2023-07-17 20:00:00|238       |
|user_393|article_189|share |2023-02-11 01:00:00|281       |
|user_17 |article_99 |share |2023-06-29 22:00:00|90        |
|user_455|article_72 |read  |2023-03-01 01:00:00|109       |
|user_418|article_100|read  |2023-11-08 22:00:00|193       |
|user_153|article_51 |like  |2023-03-28 16:00:00|71        |
|user_261|article_106|read  |2023-10-06 12:00:00|54        |
|user_399|article_20 |like  |2023-05-04 00:00:00|211       |
|user_93 |article_89 |like  |2023-11-28 10:00:00|189       |
|user_62 |article_53 |share |2023-04-17 20:00:00|209       |
|user_364|article_60 |share |2023-05-28 20:00:00|16        |
|user_314|article_138|read  |2023-03-27 06:00:00|103       |
|user_53 |article_68 |share |2023-04-18 11:00:00|216       |
|user_298|article_16 |li

In [43]:
# Data Clearning 01: Handling all datatype changes 

# # convert timestamp from string to timestamp datatype 
# # publish_date from string to Date datatype 

# we create an additional column with corrected DT and drop the incorrect column
# to_date() Column -> "publish_date" or col("publish_date") 
# withColumn function -> typecast using to_date()
# withColumn(colName: String, col: Column)

article_metadata_df = article_metadata_df.withColumn(
    "publish_date_typed", to_date(col("publish_date")) #typcased to date type 
)

article_metadata_df = article_metadata_df.drop(
    "publish_date"
).withColumnRenamed(
    "publish_date_typed",
    "publish_date"
)


In [44]:
article_metadata_df.printSchema()

root
 |-- article_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- author: string (nullable = true)
 |-- publish_date: date (nullable = true)



In [45]:
# convert timestamp from string to timestamp datatype 
# --> def to_timestamp(s: Column)

user_activity_df = user_activity_df.withColumn("timestamp_typed", to_timestamp(col("timestamp")))

In [46]:
user_activity_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp_typed: timestamp (nullable = true)



In [47]:
user_activity_df.show()

+--------+-----------+------+-------------------+----------+-------------------+
| user_id| article_id|action|          timestamp|time_spent|    timestamp_typed|
+--------+-----------+------+-------------------+----------+-------------------+
|user_372| article_73|  read|2023-07-17 20:00:00|       238|2023-07-17 20:00:00|
|user_393|article_189| share|2023-02-11 01:00:00|       281|2023-02-11 01:00:00|
| user_17| article_99| share|2023-06-29 22:00:00|        90|2023-06-29 22:00:00|
|user_455| article_72|  read|2023-03-01 01:00:00|       109|2023-03-01 01:00:00|
|user_418|article_100|  read|2023-11-08 22:00:00|       193|2023-11-08 22:00:00|
|user_153| article_51|  like|2023-03-28 16:00:00|        71|2023-03-28 16:00:00|
|user_261|article_106|  read|2023-10-06 12:00:00|        54|2023-10-06 12:00:00|
|user_399| article_20|  like|2023-05-04 00:00:00|       211|2023-05-04 00:00:00|
| user_93| article_89|  like|2023-11-28 10:00:00|       189|2023-11-28 10:00:00|
| user_62| article_53| share

In [48]:
# timestamp_typed -> timestamp

user_activity_df = user_activity_df.drop("timestamp").withColumnRenamed("timestamp_typed", "timestamp")

In [49]:
user_activity_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [50]:
# summarize your dataframes 
user_profile_df.summary().show()

# user_id has 503 non-null values 
# age has 501 non-null values  -> TODO: [number is interger type]
# region has 503 non-null values 
# gender has 503 non-null values 

+-------+-------+------------------+-------+------+
|summary|user_id|               age| region|gender|
+-------+-------+------------------+-------+------+
|  count|    503|               501|    503|   503|
|   mean|   null| 40.89021956087824|   null|  null|
| stddev|   null|13.832206047904888|   null|  null|
|    min| user_1|                18|Central|Female|
|    25%|   null|                28|   null|  null|
|    50%|   null|                40|   null|  null|
|    75%|   null|                53|   null|  null|
|    max|user_99|                64|  south|  male|
+-------+-------+------------------+-------+------+



In [51]:
article_metadata_df.summary().show()
# no null values in article 

+-------+----------+--------------------+---------+--------------+
|summary|article_id|               title| category|        author|
+-------+----------+--------------------+---------+--------------+
|  count|       202|                 202|      202|           202|
|   mean|      null|                null|     null|          null|
| stddev|      null|                null|     null|          null|
|    min| article_1|A learn meeting e...|Education|   Aaron Bates|
|    25%|      null|                null|     null|          null|
|    50%|      null|                null|     null|          null|
|    75%|      null|                null|     null|          null|
|    max|article_99|Young above centr...|   Travel|Zachary Fisher|
+-------+----------+--------------------+---------+--------------+



In [52]:
user_activity_df.summary().show()

# negative value for time spent -> -100 [TODO: ]

+-------+-------+----------+------+------------------+
|summary|user_id|article_id|action|        time_spent|
+-------+-------+----------+------+------------------+
|  count|   1001|      1001|  1001|              1001|
|   mean|   null|      null|  null|149.23276723276723|
| stddev|   null|      null|  null| 87.21324879417557|
|    min| user_1| article_1|  like|              -100|
|    25%|   null|      null|  null|                77|
|    50%|   null|      null|  null|               145|
|    75%|   null|      null|  null|               221|
|    max|user_99|article_99| share|               299|
+-------+-------+----------+------+------------------+



In [53]:
# handling null values 
# 1. put the average value 
# 2. drop the null values  
# 3. Give some filler data like NA 


# discussion 
# drop null values -> never drop the entire row when 1 value is null
# user_profile_df.filter(col("age").isNull()).show()


# put the average value 
# average age is 40 -> it does not affect the stats of age [mean will remain the same]
mean_age = int(user_profile_df.agg(avg(col("age"))).collect()[0][0])
user_profile_df = user_profile_df.fillna(mean_age, ["age"])
# it goes to your data 
# whereever it finds a null at age it will replace the null with mean_age = 40


# give some filler data 
# interger which is not a valid age -> 
# (-1 filler value) that I can filter out while performing actions on age


In [54]:
# handling mistyped value 
user_profile_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- gender: string (nullable = true)



In [55]:
# mistyped values in region and gender 
user_profile_df.groupBy("region").count().show(100, truncate=False)

+-------+-----+
|region |count|
+-------+-----+
|South  |98   |
|Central|106  |
|East   |93   |
|West   |89   |
|North  |116  |
|south  |1    |
+-------+-----+



In [56]:
# +-------+-----+
# |region |count|
# +-------+-----+
# |South  |98   |
# |Central|106  |
# |East   |93   |
# |West   |89   |
# |North  |116  |
# |south  |1    |
# +-------+-----+


# this is what we want 
# +-------+-----+
# |region |count|
# +-------+-----+
# |South  |99   |
# |Central|106  |
# |East   |93   |
# |West   |89   |
# |North  |116  |
# +-------+-----+

# mindset -> whatever can happen will happen 

user_profile_df = user_profile_df.withColumn(
    "region_typed", 
    when(col("region") == "south", "South")\
    .otherwise(col("region"))
)
user_profile_df = user_profile_df.drop("region").withColumnRenamed("region_typed", "region")

In [57]:
user_profile_df.groupBy(col("gender")).count().show()

+----------+-----+
|    gender|count|
+----------+-----+
|    Female|  109|
|    female|    1|
|     Other|  118|
|      MALE|    2|
|      male|    2|
|Non-Binary|  142|
|      Male|  129|
+----------+-----+



In [58]:
# +----------+-----+
# |    gender|count|
# +----------+-----+
# |    Female|  109|
# |    female|    1|
# |     Other|  118|
# |      MALE|    2|
# |      male|    2|
# |Non-Binary|  142|
# |      Male|  129|
# +----------+-----+


# +----------+-----+
# |    gender|count|
# +----------+-----+
# |    Female|  110|
# |     Other|  118|
# |Non-Binary|  142|
# |      Male|  133|
# +----------+-----+

user_profile_df = user_profile_df.withColumn(
    "gender_typed", 
    when(col("gender") == "female", "Female")\
    .when(col("gender") == "MALE", "Male")\
    .when(col("gender") == "male", "Male")\
    .otherwise(col("gender"))
)

user_profile_df = user_profile_df.drop("gender").withColumnRenamed("gender_typed", "gender")


In [59]:
# handling invalid data
# 1. convert the time to 0 
# 2. remove the entire row 

# define -ve time spent as invalid and all -ve should be converted to 0 

# 1. convert time to 0
# user_activity_df = user_activity_df.withColumn(
#     "time_spent_typed", 
#     when(col("time_spent") < 0, 0)\
#     .otherwise(col("time_spent"))
# )

# 2. filter()
# filters out unnecessary data from my dataset 
user_activity_df = user_activity_df.filter(col("time_spent") > 0)
user_activity_df.summary().show()

+-------+-------+----------+------+-----------------+
|summary|user_id|article_id|action|       time_spent|
+-------+-------+----------+------+-----------------+
|  count|   1000|      1000|  1000|             1000|
|   mean|   null|      null|  null|          149.482|
| stddev|   null|      null|  null|86.89950056933124|
|    min| user_1| article_1|  like|                1|
|    25%|   null|      null|  null|               77|
|    50%|   null|      null|  null|              145|
|    75%|   null|      null|  null|              221|
|    max|user_99|article_99| share|              299|
+-------+-------+----------+------+-----------------+



In [60]:
# removing duplicates by only selecting distinct 
user_profile_df = user_profile_df.distinct()
article_metadata_df = article_metadata_df.distinct()
user_activity_df = user_activity_df.distinct()

In [ ]:
# done with all data cleaning and data wrangling 

In [61]:
user_profile_df.printSchema()
article_metadata_df.printSchema()
user_activity_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- article_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- author: string (nullable = true)
 |-- publish_date: date (nullable = true)

root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [62]:
user_profile_df.show(truncate=False)
article_metadata_df.show()
user_activity_df.show(truncate=False)

+--------+---+-------+----------+
|user_id |age|region |gender    |
+--------+---+-------+----------+
|user_114|27 |West   |Non-Binary|
|user_211|39 |South  |Female    |
|user_228|63 |South  |Non-Binary|
|user_466|64 |East   |Other     |
|user_137|46 |West   |Male      |
|user_156|32 |West   |Female    |
|user_392|40 |North  |Non-Binary|
|user_438|57 |West   |Non-Binary|
|user_55 |21 |North  |Male      |
|user_140|23 |Central|Other     |
|user_349|46 |South  |Male      |
|user_8  |63 |North  |Non-Binary|
|user_79 |55 |South  |Male      |
|user_141|61 |East   |Other     |
|user_39 |30 |East   |Other     |
|user_134|43 |North  |Other     |
|user_400|52 |East   |Other     |
|user_186|28 |Central|Non-Binary|
|user_7  |35 |East   |Other     |
|user_16 |37 |West   |Other     |
+--------+---+-------+----------+
only showing top 20 rows

+-----------+--------------------+-------------+--------------------+------------+
| article_id|               title|     category|              author|publis

In [64]:
# Take these 3 tables as input 
# I want to build a health monitor 

# On the day of activity, 
# Date -> 

user_activity_df.orderBy(col("timestamp")).show(truncate=False)

+--------+-----------+------+----------+-------------------+
|user_id |article_id |action|time_spent|timestamp          |
+--------+-----------+------+----------+-------------------+
|user_466|article_177|share |204       |2023-01-01 05:00:00|
|user_457|article_61 |read  |208       |2023-01-01 05:00:00|
|user_388|article_114|like  |153       |2023-01-01 08:00:00|
|user_71 |article_111|read  |210       |2023-01-02 16:00:00|
|user_254|article_187|read  |288       |2023-01-02 23:00:00|
|user_1  |article_107|like  |107       |2023-01-03 06:00:00|
|user_306|article_164|like  |277       |2023-01-03 07:00:00|
|user_54 |article_47 |like  |144       |2023-01-03 13:00:00|
|user_283|article_111|like  |214       |2023-01-03 17:00:00|
|user_5  |article_110|share |167       |2023-01-04 01:00:00|
|user_291|article_37 |like  |179       |2023-01-04 17:00:00|
|user_69 |article_111|read  |74        |2023-01-04 22:00:00|
|user_369|article_80 |like  |145       |2023-01-05 04:00:00|
|user_5  |article_53 |li

In [ ]:
# 2023-01-01 -> action was performed by user on a given article 
# 3 unique active user, -> Daily Active Users - done

# 204 + 208 + 153, -> Overall Time spent by all Users - done

# {"share": 1, "read": 1, "like": 1}, -> user action engagement => how many actions were performed this will help you in 
# understanding how your dailyHunt is performing on a given day 


# {"centeral": 2, "north": 1}, # user region engagement => how many regions are we winning in 


# {"technology": 2} # category for most engagement 

In [72]:
# Part 1 -> getting report date, DAU, total time spent on given day 

user_activity_df.printSchema()

# |user_466|article_177|share |204       |2023-01-01 05:00:00| 2023-01-01| 1 
# |user_457|article_61 |read  |208       |2023-01-01 05:00:00| 2023-01-01| 2 
# |user_388|article_114|like  |153       |2023-01-01 08:00:00| 2023-01-01| 3

# |user_71 |article_111|read  |210       |2023-01-02 16:00:00|
# |user_254|article_187|read  |288       |2023-01-02 23:00:00|

# group by timestamp - incorrect 

# create a report_date column from timestamp 

# should we join before performing this action? 

# user_profile_df.show(truncate=False)
# article_metadata_df.show()
# user_activity_df.show(truncate=False)


# Two perspectives 
# 1st -> consider the activity anyway
    # user, Akash -> dailyHunt -> read -> deleted the profile => no proper user is present
    # user_activity_df => has the activity recorded 

# 2nd -> do not consider activity of deleted user 



# consider the activity even when user is deleted [we are not joining with any other df]

user_activity_df = user_activity_df.withColumn("report_date", to_date("timestamp"))

# opened my source code of spark in my computer and saw what should be the syntax 
daily_active_users_df = user_activity_df.groupBy("report_date").agg(
    countDistinct(col("user_id")).alias("daily_active_users")
)

# this is intermediate checking -> v.v.important 
daily_active_users_df.filter(col("report_date") == "2023-01-02").show()


root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- report_date: date (nullable = true)



+-----------+------------------+
|report_date|daily_active_users|
+-----------+------------------+
| 2023-01-02|                 2|
+-----------+------------------+



In [79]:
# Overall Time spent by all Users 

# ask questions before you start coding? 
# should we consider time spent even while liking, sharing, and reading? -> Yes
# Should we join with any other DF? -> No, since we are considering all activity as valid even if user is deactivated 


# Code the solution 
user_activity_df.printSchema()
# to get overall time spent which column is important for me ? -> time_spent
# return sum (time_spent) for a given "report_date"

total_time_spent_df = user_activity_df.groupBy(col("report_date")).agg(
    sum(col("time_spent")).alias("total_time_spent")
)
total_time_spent_df.show()


# verify if my logic is correct before moving forward ?

# |user_466|article_177|share |204       |2023-01-01 05:00:00| 2023-01-01| 1 
# |user_457|article_61 |read  |208       |2023-01-01 05:00:00| 2023-01-01| 2 
# |user_388|article_114|like  |153       |2023-01-01 08:00:00| 2023-01-01| 3

# 565 = 204 + 208 + 153

# |user_71 |article_111|read  |210       |2023-01-02 16:00:00|
# |user_254|article_187|read  |288       |2023-01-02 23:00:00|

# 498 = 210 + 288

total_time_spent_df.filter(col("report_date") == "2023-01-02").show()


root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- report_date: date (nullable = true)



+-----------+----------------+
|report_date|total_time_spent|
+-----------+----------------+
| 2023-07-15|             701|
| 2023-06-22|             240|
| 2023-09-14|             810|
| 2023-05-22|             802|
| 2023-11-08|             476|
| 2023-09-19|             127|
| 2023-06-18|             174|
| 2023-02-25|              25|
| 2023-11-22|              69|
| 2023-02-08|             438|
| 2023-11-29|             529|
| 2023-12-10|             723|
| 2023-03-12|             342|
| 2023-03-24|             379|
| 2023-09-27|             226|
| 2023-11-25|               9|
| 2023-01-01|             565|
| 2023-11-17|              72|
| 2023-07-29|             201|
| 2023-06-24|             112|
+-----------+----------------+
only showing top 20 rows



+-----------+----------------+
|report_date|total_time_spent|
+-----------+----------------+
| 2023-01-02|             498|
+-----------+----------------+



In [100]:
# user action engagement

# |user_466|article_177|share |204       |2023-01-01 05:00:00| 2023-01-01| 1 
# |user_457|article_61 |read  |208       |2023-01-01 05:00:00| 2023-01-01| 2 
# |user_388|article_114|like  |208       |2023-01-01 08:00:00| 2023-01-01| 3

# 2023-01-01, {"share": 1, "read": 1, "like": 1} 

# ask question? 
# for deleted user-> yes consider it? -> not join with our user_information 
# actual user action engagement -> should we have a threshold for like, for share and for read -> No, threshold 

# should I join with user? => no because deleted users are also considered 
user_activity_df.printSchema()

# what are the columns that are important for us 
# "report_date" and "action"
# actions -> [like, read, share]

# |user_466|article_177|1 |204       |2023-01-01 05:00:00| 2023-01-01| 1 --> 3rd when condition will be true 
# |user_457|article_61 |1  |208       |2023-01-01 05:00:00| 2023-01-01| 2 --> 2nd when condition will be true 
# |user_388|article_114|1  |208       |2023-01-01 08:00:00| 2023-01-01| 3 --> 1st when condition will be true 

user_action_engagement_df = user_activity_df.groupBy(col("report_date")).agg(
    sum(when(col("action") == "like", 1)).alias("like_count"),
    sum(when(col("action") == "read", 1)).alias("read_count"),
    sum(when(col("action") == "share", 1)).alias("share_count")
).withColumn(
    "user_engagement", 
    to_json(
        struct(
            col("like_count").alias("like"), 
            col("read_count").alias("read"), 
            col("share_count").alias("share")
        )
    )
).select(col("report_date"), col("user_engagement"))


# test with smaller set 
user_action_engagement_df.filter(col("report_date") == "2023-01-01").show(truncate=False)


root
 |-- user_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- action: string (nullable = true)
 |-- time_spent: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- report_date: date (nullable = true)



+-----------+-----------------------------+
|report_date|user_engagement              |
+-----------+-----------------------------+
|2023-01-01 |{"like":1,"read":1,"share":1}|
+-----------+-----------------------------+



In [113]:
# user region engagement => how many regions are we winning in 

# {"centeral": 2, "north": 1}, 

# 2023-01-01 -> 3 distinct distinct who are active -> how many of them were from central? 

# Ask Questions 
# I want to get for deleted users also 

# should I join with any other DF ?

# root (user_activity_df)
#  |-- user_id: string (nullable = true)
#  |-- article_id: string (nullable = true)
#  |-- action: string (nullable = true)
#  |-- time_spent: integer (nullable = true)
#  |-- timestamp: timestamp (nullable = true)
#  |-- report_date: date (nullable = true)


# root (user_profile_df)
#  |-- user_id: string (nullable = true)
#  |-- age: integer (nullable = true)
#  |-- region: string (nullable = true)
#  |-- gender: string (nullable = true)


# Step 1: Join user_activity with your user_profile based on your user_id column 
# what type of join -> inner join (theoritical: not filtering out deleted users)

joined_df = user_activity_df.join(user_profile_df, on="user_id", how="inner")
# {"East": 2, "Central": 1} 
# we should only countDistinct for each region and report_date 
# not going to do static region, I will assume there is dynamic region 

# collect_list 
user_engagement_df = joined_df.groupBy("report_date", "region").agg(
    countDistinct(col("user_id")).alias("dau") 
).groupBy("report_date").agg(
    map_from_entries(
        collect_list(
            struct(col("region"), col("dau"))
        )
    ).alias("user_region_engagement")
)


In [114]:
# Homework 
# category for most engagement 


# four dataframes 

# user_engagement_df, 

final_df = daily_active_users_df.join(
    total_time_spent_df, on="report_date", how="inner"
).join(
    user_action_engagement_df, on="report_date", how="inner"
).join(
    user_engagement_df, on="report_date", how="inner"
)

final_df.printSchema()

# report_date,
# daily_active_user
# total_time_spent 
# user_engagement
# user_action_engagement

# health monitor -> yes 

root
 |-- report_date: date (nullable = true)
 |-- daily_active_users: long (nullable = false)
 |-- total_time_spent: long (nullable = true)
 |-- user_engagement: string (nullable = true)
 |-- user_region_engagement: map (nullable = false)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = false)



In [117]:
final_df.coalesce(1).write.mode("overwrite").parquet("/user/userjuly2025019/spark/dailyHuntOutput")

In [118]:
# best practices 

# 1. Understand the use-case 
# 2. Understand the data and perform data wrangling 
# 3. Understand the output structure and ask relevant questions 
    # 3.1. For each of the step while building your business logic test with a smaller sample set 
# 4. Test the entire pipeline 
# 5. Write automated test cases 
# 6. Look if it satisfies your SLA 
# 7. Check for optimizations
# 8. Deploy
    # 8.1 create the python file 
    # 8.2 move it to your client machine 
    # 8.3 run the spark job with spark-submit command in your client 